Report 5


Anh Do

020416-2317

anhd@kth.se

In [76]:
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo
import numpy as np
import random
import itertools
from gurobipy import GRB
import gurobipy as gp
from pyomo.opt import SolverFactory

WLS = { # Please dont steal my credentials
    "WLSACCESSID": "1e6bdedb-f27d-4b0c-8a0d-24a8c94a8cb5",
    "WLSSECRET": "47ebeda5-b872-4365-9981-ef8044c28db5",
    "LICENSEID": 2707350,
}

# Problem 1

## Code

In [77]:
def build_ecp_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3)
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5:
            return pyo.Integers
        return pyo.Reals

    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=domain_rule)
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    # Linear constraint
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] <= 2)
    
    # Container for ECP cuts
    m.ecp_cuts = pyo.ConstraintList()

    return m

In [79]:
# ==========================================================
# 1. OA Model
# ==========================================================

def create_nlp_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)
    m.x = pyo.Var(m.I, bounds=lambda m, i: (-2, 2), domain=pyo.Reals)
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    # Constraints
    m.cons = pyo.ConstraintList()
    m.cons.add(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)
    m.cons.add(expr=sum(m.x[i]**2 for i in m.I) - 3 <= 0)

    return m

def create_feas_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)
    m.x = pyo.Var(m.I, bounds=lambda m, i: (-2, 2), domain=pyo.Reals)
    m.u = pyo.Var(domain=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=m.u, sense=pyo.minimize)
    
    # Relaxed Constraints
    m.cons = pyo.ConstraintList()
    m.cons.add(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= m.u)
    m.cons.add(expr=sum(m.x[i]**2 for i in m.I) - 3 <= m.u)
    return m

def create_master_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)
    
    # Integer intersection bounds {0,1,2}
    def bounds_rule(m, i):
        if i >= 5: return (0, 2) 
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5: return pyo.Integers
        return pyo.Reals

    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=domain_rule)
    m.mu = pyo.Var(domain=pyo.Reals)

    # Objective: Minimize mu
    m.obj = pyo.Objective(expr=m.mu, sense=pyo.minimize)
    
    # Support Constraint (prevents unboundedness)
    # mu >= -sum(x)
    m.obj_support = pyo.Constraint(expr=m.mu >= -sum(m.x[i] for i in m.I))
    
    m.cuts = pyo.ConstraintList()
    m.ubd_cuts = pyo.ConstraintList()
    return m

# ==========================================================
# Helpers
# ==========================================================
def fix_variables(model, y_values):
    for i in range(5, 9):
        model.x[i].fix(y_values[i])

In [84]:
grb_params = {"TimeLimit": 300, "MIPFocus": 1, "OutputFlag": 0}

with pyo.SolverFactory('gurobi_persistent', manage_env=True) as solver:
    solver.options.update(grb_params)
    
    model = build_ecp_model()
    solver.set_instance(model)
    
    tol = 1e-4
    max_iter = 1000
    iteration = 0

    print(f"{'Iter':<5} | {'Obj Value':<12} | {'Violation':<12}")
    print("-" * 40)

    while iteration < max_iter:
        iteration += 1
        
        # 1. Solve Master
        solver.solve(model)
        
        # 2. Get current values
        x_val = {i: pyo.value(model.x[i]) for i in model.I}
        current_obj = pyo.value(model.obj)
        
        # 3. Check Constraint Violation: sum(x^2) - 3 <= 0
        sum_sq = sum(x_val[i]**2 for i in model.I)
        g_val = sum_sq - 3
        
        if g_val <= tol:
            print(f"{iteration:<5} | {current_obj:<12.4f} | {g_val:<12.6f} (Converged!)")
            break
        
        print(f"{iteration:<5} | {current_obj:<12.4f} | {g_val:<12.6f}")
        
        # 4. Add Cut: g(x^k) + grad * (x - x^k) <= 0
        # g(x^k) = g_val
        # grad = 2 * x_val
        lhs = g_val + sum(2 * x_val[i] * (model.x[i] - x_val[i]) for i in model.I)
        
        model.ecp_cuts.add(lhs <= 0)
        solver.add_constraint(model.ecp_cuts[len(model.ecp_cuts)])

    print("-" * 40)
    print("Final Solution:")
    for i in model.I:
        print(f"x[{i}] = {pyo.value(model.x[i]):.4f}")

Iter  | Obj Value    | Violation   
----------------------------------------
1     | -12.0000     | 27.000000   
2     | -10.0000     | 27.000000   
1     | -12.0000     | 27.000000   
2     | -10.0000     | 27.000000   
3     | -9.5833      | 24.395833   
4     | -8.7500      | 13.062500   
3     | -9.5833      | 24.395833   
4     | -8.7500      | 13.062500   
5     | -8.7500      | 23.062500   
5     | -8.7500      | 23.062500   
6     | -8.4833      | 16.076445   
7     | -7.9500      | 9.279863    
6     | -8.4833      | 16.076445   
7     | -7.9500      | 9.279863    
8     | -7.4712      | 13.059594   
9     | -7.3015      | 6.996698    
8     | -7.4712      | 13.059594   
9     | -7.3015      | 6.996698    
10    | -6.6664      | 6.375741    
11    | -6.4261      | 10.203227   
10    | -6.6664      | 6.375741    
11    | -6.4261      | 10.203227   
12    | -6.4138      | 3.738519    
13    | -6.2752      | 4.524293    
12    | -6.4138      | 3.738519    
13    | -6.2752      | 